# LangChain + Gemini — Notes

**What is LangChain?**
An LLM alone has no memory, no access to live/private data, and can't take actions. LangChain is the toolkit that connects the LLM to prompts, parsers, tools, memory and data, so you can build real apps.

**Topics covered**
1. Setup (API key + model)
2. Calling the LLM directly
3. `ChatPromptTemplate`
4. `format_messages()` vs `prompt.invoke()`
5. Chaining with `|` (LCEL)

## 1. Setup

Install once:
```bash
pip install langchain langchain-google-genai python-dotenv
```
Put your key in a `.env` file (never hardcode it, never commit it):
```
GEMINI_API_KEY=your_key_here
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()                       # reads .env -> True if found
api_key = os.getenv("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY not found. Check your .env file."

## 2. Create the model and call it

`ChatGoogleGenerativeAI` is the LangChain wrapper for Gemini chat models.

**Note:** `invoke()` returns an `AIMessage` object. The text is in `.content`.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
)

response = llm.invoke("who won ipl 2023")
print(response.content)

> **Warning you may see:** *"Direct use of automatic function calling (AFC)... is not recommended"*.
> It's only a library warning. Your code still works. Ignore it.

## 3. ChatPromptTemplate

Instead of one plain string, we send a list of typed messages:

| Role | Meaning |
|---|---|
| `system` | tells the AI how to behave |
| `human` / `user` | the user's question or request |
| `ai` | a previous AI response (used for few-shot examples or history) |

`{topic}` is a **variable**. It gets replaced with a real value when you invoke the prompt.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Python tutor"),
    ("user", "Explain the {topic} in simple words"),
])

message = prompt.invoke({"topic": "LangChain"})
print(message)

## 4. `format_messages()` vs `prompt.invoke()`

Both fill in the variables. The only difference is **what they return**.

### `format_messages()`
Fills the variables and returns a **list of individual message objects**.

```
[SystemMessage("You are a helpful Python tutor"),
 HumanMessage("Explain the LangChain in simple words")]
```

### `prompt.invoke()`
Takes the input dict, fills the variables and returns the **complete formatted prompt** as a `ChatPromptValue`.

```
ChatPromptValue(messages=[SystemMessage(...), HumanMessage(...)])
```

### Easy way to remember
```
format_messages()  ->  list of individual messages
prompt.invoke()    ->  one complete ChatPromptValue
```

**Which one to use?**
- Use `prompt.invoke()` inside chains. It is the standard LCEL way and is what `|` calls internally.
- Use `format_messages()` when you want a plain list of messages, for example to print them or pass them to `llm.invoke(...)` directly.

In [ ]:
# format_messages -> plain list
msgs = prompt.format_messages(topic="LangChain")
print(type(msgs))
print(msgs)

# invoke -> ChatPromptValue
value = prompt.invoke({"topic": "LangChain"})
print(type(value))
print(value.to_messages())   # convert back to a list if needed

## 5. Chaining (LCEL)

The `|` operator connects steps. The output of one step becomes the input of the next:

```
prompt  ->  llm  ->  StrOutputParser  ->  transform_case
 dict      messages   AIMessage          str            str
                                        -> str           -> UPPERCASE str
```

- **`StrOutputParser`** takes the `AIMessage` and returns just the text (`.content`), so you don't write `.content` yourself.
- **Any plain function** can be part of a chain. LangChain wraps it in a `RunnableLambda` automatically. Wrapping it yourself makes it explicit.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key)
out = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Python tutor"),
    ("user", "Explain the {topic} in simple words"),
])

def transform_case(result: str) -> str:
    return result.upper()

chain = prompt | llm | out | RunnableLambda(transform_case)

In [ ]:
res = chain.invoke({"topic": "LangChain"})
print(res)

## Quick revision

- `llm.invoke("text")` returns an `AIMessage`; read the text with `.content`.
- `ChatPromptTemplate` = a reusable prompt with roles (`system`, `user`, `ai`) and `{variables}`.
- `format_messages()` returns a list of messages. `prompt.invoke()` returns a `ChatPromptValue`.
- `StrOutputParser` turns an `AIMessage` into a plain string.
- `prompt | llm | parser | function` is a chain. Run it with `chain.invoke({...})`.
- Other chain methods: `chain.stream({...})` for token-by-token output and `chain.batch([{...}, {...}])` for many inputs at once.

In [ ]:
# Bonus: streaming output (prints as it is generated)
for chunk in (prompt | llm | out).stream({"topic": "LangChain"}):
    print(chunk, end="", flush=True)